# Clean a Raw Dataset to 3 Classes (crack, corrosion, spall)

This is a **reusable template**, not a single-dataset script. It runs exploratory data analysis on a raw YOLO-format dataset, then cleans a copy of it down to the pipeline's 3 core classes (`crack`, `corrosion`, `spall`), dropping annotation-less images and images below a resolution threshold.

To use it for a new dataset, edit `DATASET_NAME` in the config cell below and rerun. It was originally built for `kelvin_bbox_all`; it's currently configured for `tf_r2`. Each rerun's parameters (dataset name, resolution threshold) should be recorded somewhere (e.g. a commit message or the README) since this notebook only reflects whichever run was last saved.

In [ ]:
from pathlib import Path
import shutil

import pandas as pd
from PIL import Image
from tqdm import tqdm

# ---- Configure per run: point this at the raw dataset you want to clean ----
DATASET_NAME = "tf_r2"  # e.g. "tf_rbb_all", "kelvin_bbox_all", "tf_r2"
DATA_DIR = Path(f"../data/data_raw/{DATASET_NAME}")
OUTPUT_DIR = Path(f"../data/data-cleaned/{DATASET_NAME}_cleaned")
# ------------------------------------------------------------------------------

IMAGES_DIR = DATA_DIR / 'images'
LABELS_DIR = DATA_DIR / 'labels'
CLASSES_PATH = DATA_DIR / 'classes.txt'

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

## 1) Exploratory Data Analysis

### a) Image-label pairing checks


In [2]:
image_files = [p for p in IMAGES_DIR.rglob('*') if p.suffix.lower() in IMAGE_EXTS]
label_files = list(LABELS_DIR.rglob('*.txt'))

image_ids = {p.stem for p in image_files}
label_ids = {p.stem for p in label_files}

images_missing_label = sorted(image_ids - label_ids)
labels_missing_image = sorted(label_ids - image_ids)

pairing_summary = pd.DataFrame({
    'item': [
        'total_image_files',
        'total_label_files',
        'images_missing_label',
        'labels_missing_image',
    ],
    'value': [
        len(image_files),
        len(label_files),
        len(images_missing_label),
        len(labels_missing_image),
    ],
})
pairing_summary


,item,value
0,total_image_files,728
1,total_label_files,728
2,images_missing_label,0
3,labels_missing_image,0


In [3]:
pd.DataFrame({'image_id': images_missing_label})


,image_id


In [4]:
pd.DataFrame({'label_id': labels_missing_image})


,label_id


### b) Overall and per-class annotation counts


In [4]:
class_names = [line.strip() for line in CLASSES_PATH.read_text().splitlines() if line.strip()]
class_counts = {i: 0 for i in range(len(class_names))}
total_annotations = 0

for label_path in label_files:
    lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
    for line in lines:
        parts = line.split()
        if not parts:
            continue
        try:
            class_id = int(parts[0])
        except ValueError:
            continue
        if class_id in class_counts:
            class_counts[class_id] += 1
            total_annotations += 1

overall_summary = pd.DataFrame({'total_annotations': [total_annotations]})
overall_summary


,total_annotations
0,4228


In [5]:
per_class_rows = []
for class_id, class_name in enumerate(class_names):
    count = class_counts.get(class_id, 0)
    percent = (count / total_annotations * 100) if total_annotations else 0
    per_class_rows.append({
        'class_id': class_id,
        'class_name': class_name,
        'annotation_count': count,
        'percent_of_total': round(percent, 2),
    })

per_class_df = pd.DataFrame(per_class_rows).sort_values(
    'annotation_count', ascending=False
).reset_index(drop=True)
per_class_df


,class_id,class_name,annotation_count,percent_of_total
0,1,crack,1232,29.14
1,8,peeling,868,20.53
2,13,water seepage,635,15.02
3,12,stains/discoloration,457,10.81
4,9,plant growth,415,9.82
5,0,corrosion,327,7.73
6,7,paint blistering,98,2.32
7,11,spall,72,1.70
8,4,dislodged / loose objects,67,1.58
9,3,damaged parts,36,0.85


### c) Distribution of annotations per image


In [7]:
label_by_id = {p.stem: p for p in label_files}
annotation_counts = []
for image_path in image_files:
    label_path = label_by_id.get(image_path.stem)
    if label_path and label_path.exists():
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        annotation_counts.append(len(lines))
    else:
        annotation_counts.append(0)

distribution = (
    pd.Series(annotation_counts)
    .value_counts()
    .sort_index()
    .reset_index()
)
distribution.columns = ['annotation_count', 'num_images']
distribution


,annotation_count,num_images
0,1,110
1,2,96
2,3,89
3,4,64
4,5,76
5,6,59
6,7,49
7,8,30
8,9,30
9,10,29


### d) Image resolution breakdown


In [8]:
resolution_counts = {}
for image_path in image_files:
    with Image.open(image_path) as img:
        width, height = img.size
    key = f'{width}x{height}'
    resolution_counts[key] = resolution_counts.get(key, 0) + 1

resolution_df = (
    pd.DataFrame(
        [{'resolution': k, 'num_images': v} for k, v in resolution_counts.items()]
    )
    .sort_values(['num_images', 'resolution'], ascending=[False, True])
    .reset_index(drop=True)
)
resolution_df


,resolution,num_images
0,8000x6000,598
1,4080x3072,65
2,4032x3024,25
3,4000x3000,13
4,5472x3078,9
5,5712x4284,9
6,1500x2000,3
7,4000x2252,3
8,3072x4080,2
9,4080x3060,1


## 2) Data Cleaning

All cleaning steps operate on a copy of the dataset. The original data remains untouched.


In [ ]:
# OUTPUT_DIR is set in the config cell above.
force_recreate = True  # Set to True to overwrite an existing cleaned copy.
skip_existing = False  # Set to True to keep existing copy without recopying.

if OUTPUT_DIR.exists():
    if skip_existing:
        print(f'{OUTPUT_DIR} already exists. Skipping copy.')
    elif force_recreate:
        shutil.rmtree(OUTPUT_DIR)
    else:
        raise FileExistsError(
            f'{OUTPUT_DIR} already exists. Set skip_existing=True to keep it or force_recreate=True to overwrite.'
        )

if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    files_to_copy = [p for p in DATA_DIR.rglob('*') if p.is_file()]
    for src_path in tqdm(files_to_copy, desc='Copying dataset', unit='file'):
        dest_path = OUTPUT_DIR / src_path.relative_to(DATA_DIR)
        dest_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_path, dest_path)
    print(f'Copied dataset to {OUTPUT_DIR}')

OUT_IMAGES_DIR = OUTPUT_DIR / 'images'
OUT_LABELS_DIR = OUTPUT_DIR / 'labels'
OUT_CLASSES_PATH = OUTPUT_DIR / 'classes.txt'


### a) Keep only crack, corrosion, spall annotations

This step also remaps class IDs to the new class list order.


In [ ]:
keep_class_names = ['corrosion', 'crack', 'spall']

original_class_names = [
    line.strip() for line in OUT_CLASSES_PATH.read_text().splitlines() if line.strip()
]
name_to_old_id = {name: idx for idx, name in enumerate(original_class_names)}

missing = [name for name in keep_class_names if name not in name_to_old_id]
if missing:
    raise ValueError(f'Missing classes in source data: {missing}')

old_id_to_new_id = {name_to_old_id[name]: new_id for new_id, name in enumerate(keep_class_names)}

filtered_label_files = list(OUT_LABELS_DIR.glob('*.txt'))
for label_path in filtered_label_files:
    kept_lines = []
    for line in label_path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        try:
            old_id = int(parts[0])
        except ValueError:
            continue
        if old_id in old_id_to_new_id:
            parts[0] = str(old_id_to_new_id[old_id])
            kept_lines.append(' '.join(parts))

    label_path.write_text('\n'.join(kept_lines) + ('\n' if kept_lines else ''))

OUT_CLASSES_PATH.write_text('\n'.join(keep_class_names) + '\n')
dropped_class_names = [name for name in original_class_names if name not in keep_class_names]
print('Label files cleaned.')
print(f'Dropped classes: {dropped_class_names}')
print(f'Updated classes.txt with: {keep_class_names}')


### b) Drop images and labels with no annotations


In [11]:
out_image_files = [p for p in OUT_IMAGES_DIR.rglob('*') if p.suffix.lower() in IMAGE_EXTS]
label_by_id = {p.stem: p for p in OUT_LABELS_DIR.glob('*.txt')}

removed_images = 0
removed_labels = 0

for image_path in out_image_files:
    label_path = label_by_id.get(image_path.stem)
    has_labels = False
    if label_path and label_path.exists():
        lines = [line for line in label_path.read_text().splitlines() if line.strip()]
        if lines:
            has_labels = True
        else:
            label_path.unlink()
            removed_labels += 1

    if not has_labels:
        image_path.unlink()
        removed_images += 1

remaining_image_ids = {p.stem for p in OUT_IMAGES_DIR.rglob('*') if p.suffix.lower() in IMAGE_EXTS}
for label_path in OUT_LABELS_DIR.glob('*.txt'):
    if label_path.stem not in remaining_image_ids:
        label_path.unlink()
        removed_labels += 1

print(f'Removed {removed_images} images and {removed_labels} labels without annotations.')


Removed 254 images and 254 labels without annotations.


### c) Drop images below a minimum resolution threshold


In [17]:
min_side_threshold = 1080  # User-defined minimum on the smaller of width/height.

out_image_files = [p for p in OUT_IMAGES_DIR.rglob('*') if p.suffix.lower() in IMAGE_EXTS]
label_by_id = {p.stem: p for p in OUT_LABELS_DIR.glob('*.txt')}

removed_images = 0
removed_labels = 0

for image_path in out_image_files:
    with Image.open(image_path) as img:
        width, height = img.size
    if min(width, height) < min_side_threshold:
        image_path.unlink()
        removed_images += 1
        label_path = label_by_id.get(image_path.stem)
        if label_path and label_path.exists():
            label_path.unlink()
            removed_labels += 1

print(f'Removed {removed_images} images and {removed_labels} labels below threshold {min_side_threshold}.')


Removed 4 images and 4 labels below threshold 1080.


## 3) Post-cleaning EDA

Re-run the EDA steps on the cleaned dataset at `OUTPUT_DIR` (set in the config cell above).


### a) Image-label pairing checks (cleaned)


In [12]:
clean_image_files = [p for p in OUT_IMAGES_DIR.rglob('*') if p.suffix.lower() in IMAGE_EXTS]
clean_label_files = list(OUT_LABELS_DIR.rglob('*.txt'))

clean_image_ids = {p.stem for p in clean_image_files}
clean_label_ids = {p.stem for p in clean_label_files}

clean_images_missing_label = sorted(clean_image_ids - clean_label_ids)
clean_labels_missing_image = sorted(clean_label_ids - clean_image_ids)

clean_pairing_summary = pd.DataFrame({
    'item': [
        'total_image_files',
        'total_label_files',
        'images_missing_label',
        'labels_missing_image',
    ],
    'value': [
        len(clean_image_files),
        len(clean_label_files),
        len(clean_images_missing_label),
        len(clean_labels_missing_image),
    ],
})
clean_pairing_summary


,item,value
0,total_image_files,474
1,total_label_files,474
2,images_missing_label,0
3,labels_missing_image,0


In [13]:
pd.DataFrame({'image_id': clean_images_missing_label})


,image_id


In [14]:
pd.DataFrame({'label_id': clean_labels_missing_image})


,label_id


### b) Overall and per-class annotation counts (cleaned)


In [15]:
clean_class_names = [line.strip() for line in OUT_CLASSES_PATH.read_text().splitlines() if line.strip()]
clean_class_counts = {i: 0 for i in range(len(clean_class_names))}
clean_total_annotations = 0

for label_path in clean_label_files:
    lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
    for line in lines:
        parts = line.split()
        if not parts:
            continue
        try:
            class_id = int(parts[0])
        except ValueError:
            continue
        if class_id in clean_class_counts:
            clean_class_counts[class_id] += 1
            clean_total_annotations += 1

clean_overall_summary = pd.DataFrame({'total_annotations': [clean_total_annotations]})
clean_overall_summary


,total_annotations
0,1631


In [16]:
clean_per_class_rows = []
for class_id, class_name in enumerate(clean_class_names):
    count = clean_class_counts.get(class_id, 0)
    percent = (count / clean_total_annotations * 100) if clean_total_annotations else 0
    clean_per_class_rows.append({
        'class_id': class_id,
        'class_name': class_name,
        'annotation_count': count,
        'percent_of_total': round(percent, 2),
    })

clean_per_class_df = (
    pd.DataFrame(clean_per_class_rows)
    .sort_values('annotation_count', ascending=False)
    .reset_index(drop=True)
)
clean_per_class_df


,class_id,class_name,annotation_count,percent_of_total
0,0,crack,1232,75.54
1,1,corrosion,327,20.05
2,2,spall,72,4.41


### c) Distribution of annotations per image (cleaned)


In [17]:
clean_label_by_id = {p.stem: p for p in clean_label_files}
clean_annotation_counts = []
for image_path in clean_image_files:
    label_path = clean_label_by_id.get(image_path.stem)
    if label_path and label_path.exists():
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        clean_annotation_counts.append(len(lines))
    else:
        clean_annotation_counts.append(0)

clean_distribution = (
    pd.Series(clean_annotation_counts)
    .value_counts()
    .sort_index()
    .reset_index()
)
clean_distribution.columns = ['annotation_count', 'num_images']
clean_distribution


,annotation_count,num_images
0,1,169
1,2,98
2,3,53
3,4,32
4,5,39
5,6,20
6,7,13
7,8,13
8,9,8
9,10,7


### d) Image resolution breakdown (cleaned)


In [24]:
clean_resolution_counts = {}
for image_path in clean_image_files:
    with Image.open(image_path) as img:
        width, height = img.size
    key = f'{width}x{height}'
    clean_resolution_counts[key] = clean_resolution_counts.get(key, 0) + 1

clean_resolution_df = (
    pd.DataFrame(
        [{'resolution': k, 'num_images': v} for k, v in clean_resolution_counts.items()]
    )
    .sort_values(['num_images', 'resolution'], ascending=[False, True])
    .reset_index(drop=True)
)
clean_resolution_df


,resolution,num_images
0,5280x3956,1967
1,8000x6000,467
2,5280x2970,129
3,4032x3024,27
4,4000x3000,24
5,3024x4032,6
6,5712x4284,4
7,4284x5712,2
8,8160x6120,1


## 4) Output

Cleaned data lives wherever `OUTPUT_DIR` points (set in the config cell above), with `images/`, `labels/`, and `classes.txt`.
